# 🏗️ PROYECTO FINAL: "Asistente Educativo sobre el Mercado de Capitales"
# RAG local completo con interfaz Gradio


## Integrantes:

### * GUZMÁN, CYNTHIA
### * NAVAS, JOSÉ

---
## Objetivo

Ensamblar un sistema RAG completo que funcione desde la terminal: cargás tus propios PDFs, los vectorizás, los almacenás en ChromaDB, consultás con un LLM local y todo queda empaquetado en una interfaz Gradio lista para deployar en HuggingFace Spaces.
El objetivo de este trabajo es desarrollar un asistente educativo basado en la técnica RAG  que permita a usuarios principiantes consultar y comprender conceptos fundamentales del mercado de capitales mediante preguntas en lenguaje natural. El sistema utiliza documentación especializada para recuperar información relevante y generar respuestas claras y accesibles sobre instrumentos financieros, perfiles de inversor, mercado bursátil y otros conceptos relacionados.
El objetivo principal es facilitar el acceso al conocimiento financiero, evitando que los usuarios deban revisar manuales, cursos o documentos extensos para obtener definiciones básicas y explicaciones sobre la terminología utilizada en el mercado de capitales. El asistente tiene un fin exclusivamente educativo e informativo, sin brindar recomendaciones de inversión ni asesoramiento financiero personalizado.


## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Pipeline end-to-end** | Sistema completo donde cada pieza pasa su salida a la siguiente sin intervención manual. |
| **ChromaDB persistente** | Base de datos vectorial que guarda los embeddings en disco para no recalcularlos en cada ejecución. |
| **Gradio** | Librería que convierte funciones Python en interfaces web interactivas con pocas líneas de código. |
| **HuggingFace Spaces** | Plataforma gratuita que permite publicar aplicaciones Gradio en la web con un clic. |
| **Serverless inference** | Modelo de IA que corre en la infraestructura de HuggingFace sin que vos alojes nada. |
| **app.py** | Archivo principal de una aplicación Gradio — el punto de entrada que Spaces busca para arrancar. |

## 1- Instalación de dependencias

In [1]:
# Instalamos todas las dependencias del proyecto integrador
!pip install langchain langchain-community langchain-chroma langchain-ollama \
    langchain-text-splitters langchain-core \
    chromadb sentence-transformers \
    gradio pypdf python-dotenv -q

print("✓ Dependencias instaladas")
print("  Recordá tener Ollama corriendo: ollama serve")

✓ Dependencias instaladas
  Recordá tener Ollama corriendo: ollama serve



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2- Configuración del entorno

Para la implementación del modelo de lenguaje se analizaron distintas alternativas disponibles en Ollama. Entre ellas se consideró utilizar modelos más grandes, como Llama 3.2 3B, que suelen ofrecer respuestas más completas debido a su mayor cantidad de parámetros. Sin embargo, el proyecto se desarrolló en una computadora sin GPU dedicada, por lo que era necesario utilizar un modelo que pudiera ejecutarse de forma eficiente con los recursos disponibles.

Por este motivo se seleccionó Gemma 3 1B, ya que ofreció un buen equilibrio entre consumo de recursos, velocidad de respuesta y calidad de las respuestas obtenidas. Durante las pruebas realizadas, el modelo logró responder correctamente consultas relacionadas con conceptos del mercado de capitales, como acciones, bonos, CEDEAR, perfiles de inversor y riesgo financiero, manteniendo tiempos de respuesta adecuados para una aplicación local.

Asimismo, se configuró el parámetro num_ctx para definir la cantidad de contexto que el modelo puede procesar en cada consulta. Esta configuración permitió que los fragmentos recuperados por el sistema RAG fueran utilizados de manera eficiente sin exceder las capacidades del equipo empleado.

En conclusión, Gemma 3 1B fue elegido por ser el modelo que mejor se adaptó al hardware disponible, permitiendo ejecutar el asistente de forma local y obtener respuestas satisfactorias sin necesidad de infraestructura especializada.

In [2]:
import platform
import subprocess

# Detectamos el sistema operativo y la arquitectura del procesador
sistema = platform.system()     # 'Darwin', 'Linux', 'Windows'
maquina = platform.machine()    # 'arm64' (Apple Silicon), 'x86_64'

# Verificamos si hay GPU NVIDIA disponible (solo Linux/Windows)
tiene_cuda = False
if sistema in ("Linux", "Windows"):
    try:
        resultado = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        tiene_cuda = resultado.returncode == 0
    except FileNotFoundError:
        tiene_cuda = False

# Determinamos el perfil de hardware y configuramos el modelo en consecuencia
es_apple_silicon = (sistema == "Darwin" and maquina == "arm64")

if es_apple_silicon:
    PERFIL_HARDWARE = "Apple Silicon (Metal)"
    MODEL_NAME      = "gemma3:1b"   # liviano, corre bien en M-series
    NUM_CTX         = 4096
elif tiene_cuda:
    PERFIL_HARDWARE = "NVIDIA GPU (CUDA)"
    MODEL_NAME      = "granite4:latest"
    NUM_CTX         = 8192
else:
    PERFIL_HARDWARE = "CPU (sin GPU dedicada)"
    MODEL_NAME      = "gemma3:1b"   # el más conservador
    NUM_CTX         = 1024

print(f"Sistema:          {sistema} — {maquina}")
print(f"Perfil detectado: {PERFIL_HARDWARE}")
print(f"Modelo elegido:   {MODEL_NAME}")
print(f"Contexto (tokens):{NUM_CTX}")

Sistema:          Windows — AMD64
Perfil detectado: CPU (sin GPU dedicada)
Modelo elegido:   gemma3:1b
Contexto (tokens):1024


In [3]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Carpeta donde el sistema buscará los PDFs por defecto
CARPETA_DATOS = os.path.join(os.getcwd(), "CARPETA_DATOS")

# Directorio donde ChromaDB guardará los vectores en disco
DIRECTORIO_CHROMA = os.path.join(os.getcwd(), "chroma_proyecto")

print(f"Carpeta de PDFs:  {CARPETA_DATOS}")
print(f"Base vectorial:   {DIRECTORIO_CHROMA}")

Carpeta de PDFs:  c:\Tecnicas_habla\ifts24-lab-pln-2026\001\003-LAB\Trabajo_practico Final integrador\CARPETA_DATOS
Base vectorial:   c:\Tecnicas_habla\ifts24-lab-pln-2026\001\003-LAB\Trabajo_practico Final integrador\chroma_proyecto


## 3- Carga de documentos PDF



In [4]:
from langchain_community.document_loaders import PyPDFLoader

# Listamos todos los PDFs en la carpeta de datos
archivos_pdf = sorted([
    f for f in os.listdir(CARPETA_DATOS)
    if f.lower().endswith(".pdf")
])

if not archivos_pdf:
    print("⚠️  No hay PDFs en la carpeta.")
    print(f"   Copiá al menos un PDF en: {CARPETA_DATOS}")
else:
    print(f"Archivos encontrados: {len(archivos_pdf)}")

Archivos encontrados: 16


In [7]:
# Cargamos página por página y acumulamos en una lista
documentos_crudos = []

for nombre_archivo in archivos_pdf:
    ruta_completa = os.path.join(CARPETA_DATOS, nombre_archivo)
    loader = PyPDFLoader(ruta_completa)
    paginas = loader.load()
    documentos_crudos.extend(paginas)
    print(f"  ✓ {nombre_archivo} — {len(paginas)} páginas")

print(f"\n✓ Total: {len(documentos_crudos)} páginas cargadas")

  ✓ 2023-02-Curso-estudiantes-universitarios-clase-3.pdf — 59 páginas
  ✓ Analisis de Bonos.pdf — 61 páginas
  ✓ Finanzas-Basicas-Encuentro-4.pdf — 34 páginas
  ✓ Glosario de Bonos.pdf — 6 páginas
  ✓ Glosario de acciones.pdf — 3 páginas
  ✓ Glosario de opciones.pdf — 3 páginas
  ✓ Guia proteccionista para inversores.pdf — 33 páginas
  ✓ Haciendo-crecer-su-dinero.pdf — 9 páginas
  ✓ Importancia del mercado de capitales.pdf — 84 páginas
  ✓ Indice de liquidez y calidad crediticia de fondos comunes de inversión.pdf — 53 páginas
  ✓ Introduccion-al-mercado-de-capitales.pdf — 198 páginas
  ✓ ManualUniversitarios.pdf — 112 páginas
  ✓ Manual_de_Capacitacion.pdf — 177 páginas
  ✓ Mercado de capitales y sistemas financieros.pdf — 8 páginas
  ✓ Modulo-Economia-y-M.C.-Curso-Introduccion-al-M.C.pdf — 48 páginas
  ✓ Porfolios Financieros.pdf — 32 páginas

✓ Total: 920 páginas cargadas


## 4- Fragmentación de documentos
### Tamaño de los chunks 
Durante el desarrollo de este RAG se analizaron distintos tamaños de chunk para evaluar cómo afectaban la recuperación de información y la calidad de las respuestas generadas.
En una primera instancia se utilizaron chunks muy pequeños, de aproximadamente 50 caracteres. Se observó que gran parte de los conceptos financieros quedaban fragmentados en múltiples partes. Definiciones como "acción", "bono", "CEDEAR" o "caución" aparecían divididas en varios fragmentos, dificultando la recuperación de una explicación completa y reduciendo el contexto disponible para el modelo.
Posteriormente se evaluó el efecto de utilizar chunks muy grandes. En este caso, aunque se conservaba más contexto, los fragmentos comenzaban a incluir varios temas financieros simultáneamente. Por ejemplo, dentro de un mismo chunk podían aparecer conceptos relacionados con acciones, bonos, perfiles de inversor y fondos comunes de inversión. Esto provocaba que algunas respuestas incorporaran información menos relevante o mezclaran conceptos diferentes.
A partir de estas observaciones se optó por utilizar un tamaño de chunk intermedio de 600 caracteres con un solapamiento de 80 caracteres. Esta configuración permitió mantener suficiente contexto para comprender correctamente los conceptos financieros, evitando al mismo tiempo la fragmentación excesiva o la mezcla de temas. Como resultado, el sistema logró responder de manera satisfactoria consultas sobre acciones, bonos, CEDEARs, cauciones, fondos comunes de inversión y otros conceptos básicos del mercado de capitales.
### chunk_overlap
Se observó que algunas definiciones financieras quedaban divididas entre dos fragmentos consecutivos. Cuando esto ocurría, el recuperador podía encontrar solo una parte de la definición, perdiendo información importante para generar una respuesta completa.
Por ejemplo, en conceptos como fondos comunes de inversión o mercado de capitales, una parte de la explicación podía quedar al final de un chunk y el resto al comienzo del siguiente. Si no existiera superposición (chunk_overlap = 0), el sistema podría recuperar únicamente uno de esos fragmentos y perder parte del contexto.
Por este motivo se utilizó un chunk_overlap de 80 caracteres. Gracias a esta superposición, las ideas que se encuentran cerca de los límites de los fragmentos aparecen parcialmente repetidas en el chunk siguiente, preservando la continuidad semántica del texto.
La elección de 80 caracteres resultó adecuada para el tamaño de chunk utilizado (600 caracteres), ya que permitió mantener el contexto de las definiciones sin generar una cantidad excesiva de texto repetido dentro de la base vectorial.
En consecuencia, el sistema logró recuperar definiciones más completas y generar respuestas más coherentes sobre conceptos del mercado de capitales.
#### Por lo tanto: 
No usamos chunk_overlap por redundancia, sino para preservar contexto semántico entre fragmentos consecutivos. Sin overlap, las ideas que quedan en los bordes de los chunks pueden romperse y degradar la calidad de la búsqueda vectorial. Con un overlap moderado (10%-20%), el sistema RAG recupera información más relevante y genera respuestas más precisas y coherentes.
#### ¿Cuánto overlap usar?
Una regla práctica muy utilizada es:
| Chunk Size | Overlap recomendado |
| ---------- | ------------------- |
| 300        | 30–50               |
| 500        | 50–100              |
| 1000       | 100–200             |
| 2000       | 200–400             |
- Ejemplo con texto del dominio financiero del PDF mostrando exactamente cómo una idea sobre "fondos comunes de inversión" quedaría cortada sin overlap
- Muestra los 80 caracteres de solapamiento que salvan la continuidad semántica entre fragmentos
- Regla práctica calibrada al tipo de texto del proyecto


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter intenta dividir por párrafos primero,
# luego por oraciones, luego por espacios — respetando la coherencia semántica
divisor = RecursiveCharacterTextSplitter(
    chunk_size=600,        # máximo 600 caracteres por fragmento
    chunk_overlap=80,      # 80 caracteres de solapamiento entre fragmentos
    separators=["\n\n", "\n", ". ", " "]
)

fragmentos = divisor.split_documents(documentos_crudos)

print(f"✓ Fragmentación completada")
print(f"  Documentos originales: {len(documentos_crudos)} páginas")
print(f"  Fragmentos generados:  {len(fragmentos)}")

# Mostramos un fragmento de ejemplo para verificar
print(f"\n◈ Ejemplo de fragmento:")
print(f"  Fuente: {fragmentos[0].metadata.get('source', 'desconocida')}")
print(f"  Página: {fragmentos[0].metadata.get('page', '?')}")
print(f"  Texto:  {fragmentos[0].page_content[:200]}...")

✓ Fragmentación completada
  Documentos originales: 920 páginas
  Fragmentos generados:  3073

◈ Ejemplo de fragmento:
  Fuente: c:\Tecnicas_habla\ifts24-lab-pln-2026\001\003-LAB\Trabajo_practico Final integrador\CARPETA_DATOS\2023-02-Curso-estudiantes-universitarios-clase-3.pdf
  Página: 0
  Texto:  Curso introducción al Mercado 
de Capitales
para estudiantes universitarios...


## 4- Embeddings y ChromaDB persistente
En esta etapa se transformó el contenido de los documentos PDF en embeddings, es decir, vectores numéricos que representan el significado semántico del texto. Esto permite que el sistema no busque únicamente palabras exactas, sino también conceptos relacionados.
Una vez generados los embeddings, se almacenaron en una base vectorial utilizando ChromaDB. La función de esta base es guardar los vectores y recuperar los fragmentos más relevantes cuando el usuario realiza una consulta.
Además, se utilizó una configuración persistente, lo que significa que los embeddings quedan almacenados en disco y no necesitan recalcularse cada vez que se ejecuta la aplicación. Esto reduce significativamente los tiempos de carga y mejora el rendimiento general del sistema.
Gracias a esta etapa, cuando el usuario realiza una pregunta como "¿Qué es una acción?" o "¿Qué es una caución?", el sistema puede localizar los fragmentos más relevantes de los documentos antes de enviarlos al modelo de lenguaje para generar la respuesta.


In [9]:
from langchain_community.embeddings import SentenceTransformerEmbeddings

# multilingual-e5-large corre localmente sin consumir API
# Entiende bien el español rioplatense y múltiples idiomas
modelo_embeddings = SentenceTransformerEmbeddings(
    model_name="intfloat/multilingual-e5-large"
)

print("✓ Modelo de embeddings configurado")
print("  intfloat/multilingual-e5-large — local, sin API")

C:\Users\cynth\AppData\Local\Temp\ipykernel_19832\1228755670.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  modelo_embeddings = SentenceTransformerEmbeddings(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Modelo de embeddings configurado
  intfloat/multilingual-e5-large — local, sin API


In [10]:
from langchain_chroma import Chroma

# Creamos (o recargamos si ya existe) la base vectorial persistente
# Si el directorio chroma_proyecto ya existe, Chroma lo carga sin recalcular
vectorstore = Chroma.from_documents(
    documents=fragmentos,
    embedding=modelo_embeddings,
    collection_name="proyecto_rag",
    persist_directory=DIRECTORIO_CHROMA
)

print(f"✓ Base vectorial lista")
print(f"  Directorio: {DIRECTORIO_CHROMA}")
print(f"  Fragmentos indexados: {len(fragmentos)}")

✓ Base vectorial lista
  Directorio: c:\Tecnicas_habla\ifts24-lab-pln-2026\001\003-LAB\Trabajo_practico Final integrador\chroma_proyecto
  Fragmentos indexados: 3073


## 5- LLM local con Ollama
En esta etapa se configuró el modelo de lenguaje local utilizando Ollama. Se eligió Gemma 3 1B porque podía ejecutarse correctamente en el hardware disponible, sin necesidad de una GPU dedicada, manteniendo un buen equilibrio entre velocidad y calidad de las respuestas.
También se configuró el parámetro temperature = 0.1, con el objetivo de obtener respuestas más estables y precisas, reduciendo la generación de información no presente en los documentos.
Por otro lado, se definió el parámetro num_ctx, que indica la cantidad de contexto que el modelo puede procesar en cada consulta. Esto permite que el modelo utilice los fragmentos recuperados por ChromaDB para responder las preguntas del usuario de forma coherente.


In [11]:
from langchain_ollama import OllamaLLM

# Conectamos con el servidor Ollama que corre localmente en el puerto 11434
# temperature=0.1 para respuestas precisas y reproducibles
llm = OllamaLLM(
    model=MODEL_NAME,
    temperature=0.1,
    num_ctx=NUM_CTX,
)

# Verificamos que Ollama responde antes de armar el pipeline completo
respuesta_prueba = llm.invoke("Respondé solo con 'ok' si estás funcionando.")
print(f"✓ Ollama responde: {respuesta_prueba.strip()}")
print(f"  Modelo activo: {MODEL_NAME}")

✓ Ollama responde: Ok.
  Modelo activo: gemma3:1b


## 6- Pipeline RAG con LCEL
La función formatear_documentos() actúa como un adaptador entre el retriever y el prompt. ChromaDB devuelve una lista de objetos Document, mientras que el PromptTemplate necesita recibir texto plano. Por este motivo, la función extrae el contenido (page_content) de cada fragmento recuperado y los concatena en un único bloque de texto que luego es enviado al modelo. Sin esta transformación, el pipeline no podría construir correctamente el contexto para generar la respuesta.

El parámetro k define cuántos fragmentos recupera ChromaDB antes de enviarlos al modelo de lenguaje.
Durante las pruebas del proyecto se evaluaron distintos valores de k. Con k=3, las respuestas eran rápidas, pero en algunos casos resultaban incompletas cuando la información estaba distribuida en varios fragmentos o documentos. Posteriormente se probó con k=6, observándose que el modelo recibía una cantidad mayor de contexto, pero también comenzaban a aparecer fragmentos menos relevantes, generando respuestas más extensas y con mayor ruido informativo.
Finalmente se seleccionó k=5, ya que representó un equilibrio adecuado entre cantidad de información recuperada, precisión de las respuestas y tiempo de procesamiento. Con este valor el sistema logró responder correctamente consultas sobre conceptos financieros como acciones, bonos, CEDEAR, cauciones, perfiles de inversor y renta fija o variable, manteniendo respuestas claras y relevantes.
Por lo tanto, se concluyó que k=5 era la configuración más adecuada para el conjunto de documentos utilizado, ya que ofrecía una cobertura suficiente de información sin incorporar un exceso de contexto que pudiera afectar la calidad de las respuestas

In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# El retriever busca los 5 fragmentos más similares a la pregunta
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

def formatear_documentos(docs):
    # Une los fragmentos recuperados en un solo bloque de texto
    return "\n\n".join(doc.page_content for doc in docs)

TEMPLATE = """Sos un asistente especializado en mercado de capitales e inversiones para principiantes.

Tu tarea es responder utilizando únicamente la información presente en los documentos recuperados.

REGLAS IMPORTANTES:

- No inventes información.
- No agregues conceptos que no aparezcan en los documentos.
- Si el documento enumera elementos, respetá exactamente la cantidad indicada.
- Si encontrás información parcial, respondé con lo que encontraste de forma clara.
- Si la información es insuficiente para responder correctamente, indicá:
  "No encontré información suficiente en los documentos cargados."
- Explicá los conceptos de forma sencilla y educativa.
- No brindes recomendaciones personalizadas de inversión.
- No recomiendes comprar ni vender activos financieros.



Documentos:
{context}

Pregunta: {question}

Respuesta:"""

prompt = PromptTemplate(
    template=TEMPLATE,
    input_variables=["context", "question"]
)

pipeline_rag = (
    {"context": retriever | formatear_documentos, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✓ Pipeline RAG configurado")
print("  Flujo: pregunta → ChromaDB (k=5) → prompt → Ollama → respuesta")

✓ Pipeline RAG configurado
  Flujo: pregunta → ChromaDB (k=5) → prompt → Ollama → respuesta


In [30]:
# Probamos el pipeline con una pregunta de prueba antes de abrir la interfaz
pregunta_test = "¿Qué es una acción?"

respuesta = pipeline_rag.invoke(pregunta_test)  # ← único cambio


print(f"Pregunta: {pregunta_test}")
print(f"\nRespuesta:")
print(respuesta)

Pregunta: ¿Qué es una acción?

Respuesta:
Una acción es cada una de las partes en que se di-
vide el capital de las empresas constituidas jurídica-
mente bajo la forma de sociedad anónima.



In [22]:
pregunta = "¿Qué es una caución?"

docs = retriever.invoke(pregunta)

for i, doc in enumerate(docs[:3], 1):
    print(f"\n===== DOCUMENTO {i} =====")
    print(doc.page_content)


===== DOCUMENTO 1 =====
donde la parte superavitaria de fondos presta un determinado capital (lado colocador de fondos) a 
la unidad deficitaria de fondos (tomador  de los fondos), quien pide prestado el monto convenido, 
pero con la obligación de colocar títulos en garantía para garantizar el cumplimiento de la 
devolución del capital más los intereses correspondientes. Las cauciones bursátiles tienen u na 
rentabilidad anual proporcional al plazo colocado y una fecha de vencimiento estipulada ex-ante del 
compromiso del contrato. La caución se caracteriza por ser una alternativa de colocación de fondos

===== DOCUMENTO 2 =====
títulos otorgan el derecho a percibir intereses anuales y la  devolución del capital invertido al 
vencimiento. Este mecanismo de financiamiento es particularmente importante para las empresas 
derivado del beneficio impositivo de contraer deuda, dado que los intereses anuales que la empresa 
paga a sus inversores  disminuyen el resultado del ejercicio, reduci

In [24]:
pregunta = "¿Qué es una caución?"

respuesta = pipeline_rag.invoke(pregunta)

print(respuesta)

Una caución es un préstamo entre partes en el marco del mercado de capitales, que permite percibir intereses anuales y la devolución del capital invertido al vencimiento. Es una alternativa de colocación de fondos títulos.



## 7- Interfaz Gradio

En este proyecto los fragmentos se acumulan. Esto se debe a que se utiliza vectorstore.add_documents(nuevos_fragmentos), que agrega los nuevos documentos al vectorstore existente sin eliminar los anteriores. De esta manera, la base de conocimiento crece a medida que se incorporan nuevos PDFs y las consultas pueden recuperar información tanto de los documentos originales como de los agregados posteriormente.

Mostrar las fuentes permite verificar de dónde proviene la información utilizada para generar la respuesta. Esto aporta transparencia al sistema, facilita la validación de los resultados y ayuda a detectar posibles errores o respuestas incompletas. Además, permite al usuario consultar directamente el documento y la página de donde se obtuvo la información.

Durante las pruebas realizadas se observó que el sistema puede recuperar información aun cuando la consulta y los documentos se encuentran en idiomas diferentes. Por ejemplo, se realizaron preguntas en inglés como "What is financial risk?" y "What is a CEDEAR?" sobre documentos cargados en español, obteniéndose respuestas correctas. Esto indica que los embeddings utilizados capturan similitudes semánticas más allá de la coincidencia exacta de palabras.
Sin embargo, también se detectaron limitaciones. En la consulta "What is a caución?" el sistema generó una respuesta incorrecta, diferente a la definición presente en los documentos. Esto demuestra que, aunque el sistema tiene capacidad multilingüe, la calidad de la recuperación puede verse afectada cuando se utilizan conceptos muy específicos o propios del mercado argentino. Por esta razón, los mejores resultados se obtuvieron cuando tanto los documentos como las consultas se realizaron en español.


In [31]:
import gradio as gr

# ─── Funciones que conectan la interfaz con el pipeline ───────────────────────

def cargar_pdfs_interfaz(archivos):
    """Recibe archivos subidos desde Gradio, los indexa y devuelve un mensaje de estado."""
    if not archivos:
        return "No se seleccionaron archivos."

    nuevas_paginas = []
    nombres = []

    for archivo in archivos:
        loader = PyPDFLoader(archivo.name)
        paginas = loader.load()
        nuevas_paginas.extend(paginas)
        nombres.append(Path(archivo.name).name)

    nuevos_fragmentos = divisor.split_documents(nuevas_paginas)

    # Agregamos al vectorstore existente sin reiniciarlo
    vectorstore.add_documents(nuevos_fragmentos)

    return f"✓ Archivos cargados: {', '.join(nombres)}\n✓ Fragmentos indexados: {len(nuevos_fragmentos)}"


def responder_pregunta(pregunta, historial):
    """Invoca el pipeline RAG y devuelve la respuesta junto con las fuentes consultadas."""
    if not pregunta.strip():
        return historial, ""

    respuesta = pipeline_rag.invoke(pregunta)


    # Recuperamos los fragmentos fuente para mostrarlos
    fragmentos_fuente = retriever.invoke(pregunta)

    lineas_fuente = []
    for frag in fragmentos_fuente:
        fuente = Path(frag.metadata.get("source", "desconocida")).name
        pagina = frag.metadata.get("page", "?")
        lineas_fuente.append(f"• {fuente} (pág. {pagina})")

    # Gradio 5 usa formato de mensajes (OpenAI-style), no tuplas
    historial = historial + [
        {"role": "user",      "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    texto_fuentes = "\n".join(lineas_fuente)

    return historial, texto_fuentes


print("✓ Funciones de interfaz definidas")

✓ Funciones de interfaz definidas


In [32]:
# ─── Construcción de la interfaz ──────────────────────────────────────────────

with gr.Blocks(title="RAG Local — IFTS24", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# RAG Local con Ollama y ChromaDB")
    gr.Markdown("**Laboratorio de PLN — IFTS24, 2026**")

    with gr.Tab("📄 Cargar documentos"):
        gr.Markdown("Subí uno o más PDFs para agregarlos a la base de conocimiento.")
        upload_component = gr.File(
            label="Seleccioná tus PDFs",
            file_types=[".pdf"],
            file_count="multiple"
        )
        boton_cargar = gr.Button("Indexar documentos", variant="primary")
        estado_carga = gr.Textbox(label="Estado", interactive=False, lines=3)
        boton_cargar.click(
            fn=cargar_pdfs_interfaz,
            inputs=[upload_component],
            outputs=[estado_carga]
        )

    with gr.Tab("💬 Hacer preguntas"):
        # Gradio 6.x: el formato role/content es el único disponible, sin parámetro type
        chatbot_componente = gr.Chatbot(label="Conversación", height=400)
        with gr.Row():
            pregunta_componente = gr.Textbox(
                label="Tu pregunta",
                placeholder="¿Qué dice el documento sobre...?",
                scale=4
            )
            boton_preguntar = gr.Button("Preguntar", variant="primary", scale=1)
        fuentes_componente = gr.Textbox(
            label="Fragmentos consultados",
            interactive=False,
            lines=3
        )
        boton_preguntar.click(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )
        pregunta_componente.submit(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )

demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


---

## 🌐 Hacia HuggingFace Spaces

Hasta acá el sistema corre en tu máquina con Ollama. Para publicarlo en HuggingFace Spaces necesitamos dos cambios:

1. **LLM**: Ollama no corre en Spaces — usamos la API de inferencia serverless de HuggingFace (gratis con una cuenta).
2. **ChromaDB**: usamos modo en memoria (sin `persist_directory`) porque el almacenamiento en Spaces es efímero.

Las celdas siguientes generan el `app.py` y el `requirements.txt` listos para subir.

### Pasos para publicar

1. Creá una cuenta en [huggingface.co](https://huggingface.co) si no tenés.
2. Generá un token en `Settings → Access Tokens` (con permisos de escritura).
3. En tu perfil, creá un nuevo Space → tipo **Gradio**.
4. Subí los archivos `app.py` y `requirements.txt` generados abajo.
5. En la sección `Settings → Repository Secrets` del Space, creá el secreto `HF_TOKEN` con tu token.
6. El Space se construye automáticamente y queda disponible en una URL pública.

In [35]:
%%writefile app.py
# ─────────────────────────────────────────────────────────────────────────────
# app.py — RAG con HuggingFace Inference API y Gradio
# Para HuggingFace Spaces: configurá el secreto HF_TOKEN en Settings.
# Uso local:  HF_TOKEN=tu_token python app.py
# ─────────────────────────────────────────────────────────────────────────────

import os
from pathlib import Path
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEndpoint

# ─── Configuración ────────────────────────────────────────────────────────────

HF_TOKEN  = os.environ.get("HF_TOKEN", "")
MODEL_ID  = "meta-llama/Llama-3.2-3B-Instruct"  # gratuito con token HF

if not HF_TOKEN:
    raise ValueError("Configurá el secreto HF_TOKEN en el Space.")

# ─── Embeddings locales (corren en la CPU del Space) ──────────────────────────

modelo_embeddings = SentenceTransformerEmbeddings(
    model_name="intfloat/multilingual-e5-large"
)

# ─── ChromaDB en memoria (sin disco para Spaces) ──────────────────────────────

vectorstore = Chroma(
    collection_name="proyecto_rag_spaces",
    embedding_function=modelo_embeddings
)

# ─── Divisor de texto ─────────────────────────────────────────────────────────

divisor = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "]
)

# ─── LLM via HuggingFace Serverless Inference ─────────────────────────────────

llm = HuggingFaceEndpoint(
    repo_id=MODEL_ID,
    temperature=0.1,
    max_new_tokens=512,
    huggingfacehub_api_token=HF_TOKEN
)

# ─── Pipeline RAG ─────────────────────────────────────────────────────────────

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def formatear_documentos(docs):
    return "\n\n".join(doc.page_content for doc in docs)

TEMPLATE = """Respondé la siguiente pregunta usando ÚNICAMENTE los documentos proporcionados.
Si la respuesta no está, decilo claramente.

Documentos:
{context}

Pregunta: {question}

Respuesta:"""

prompt = PromptTemplate(
    template=TEMPLATE,
    input_variables=["context", "question"]
)

pipeline_rag = (
    {"context": retriever | formatear_documentos, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ─── Funciones de la interfaz ─────────────────────────────────────────────────

def cargar_pdfs_interfaz(archivos):
    if not archivos:
        return "No se seleccionaron archivos."
    nuevas_paginas = []
    nombres = []
    for archivo in archivos:
        loader = PyPDFLoader(archivo.name)
        paginas = loader.load()
        nuevas_paginas.extend(paginas)
        nombres.append(Path(archivo.name).name)
    nuevos_fragmentos = divisor.split_documents(nuevas_paginas)
    vectorstore.add_documents(nuevos_fragmentos)
    return f"✓ Archivos: {', '.join(nombres)}\n✓ Fragmentos: {len(nuevos_fragmentos)}"

def responder_pregunta(pregunta, historial):
    if not pregunta.strip():
        return historial, ""
    respuesta = pipeline_rag.invoke(pregunta)
    fragmentos_fuente = retriever.invoke(pregunta)
    lineas_fuente = []
    for frag in fragmentos_fuente:
        fuente = Path(frag.metadata.get("source", "desconocida")).name
        pagina = frag.metadata.get("page", "?")
        lineas_fuente.append(f"• {fuente} (pág. {pagina})")
    historial = historial + [
        {"role": "user",      "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    return historial, "\n".join(lineas_fuente)

# ─── Interfaz Gradio ──────────────────────────────────────────────────────────

with gr.Blocks(title="RAG Local — IFTS24", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# RAG con HuggingFace Spaces")
    gr.Markdown("**Laboratorio de PLN — IFTS24, 2026**")

    with gr.Tab("📄 Cargar documentos"):
        upload_component = gr.File(
            label="Seleccioná tus PDFs",
            file_types=[".pdf"],
            file_count="multiple"
        )
        boton_cargar = gr.Button("Indexar documentos", variant="primary")
        estado_carga = gr.Textbox(label="Estado", interactive=False, lines=3)
        boton_cargar.click(
            fn=cargar_pdfs_interfaz,
            inputs=[upload_component],
            outputs=[estado_carga]
        )

    with gr.Tab("💬 Hacer preguntas"):
        chatbot_componente = gr.Chatbot(label="Conversación", height=400)
        with gr.Row():
            pregunta_componente = gr.Textbox(
                label="Tu pregunta",
                placeholder="¿Qué dice el documento sobre...?",
                scale=4
            )
            boton_preguntar = gr.Button("Preguntar", variant="primary", scale=1)
        fuentes_componente = gr.Textbox(
            label="Fragmentos consultados",
            interactive=False,
            lines=3
        )
        boton_preguntar.click(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )
        pregunta_componente.submit(
            fn=responder_pregunta,
            inputs=[pregunta_componente, chatbot_componente],
            outputs=[chatbot_componente, fuentes_componente]
        )

demo.launch()

Writing app.py


In [36]:
%%writefile requirements.txt
# Proyecto Final — RAG con Gradio
# Laboratorio de PLN — IFTS24, 2026
#
# HuggingFace Spaces instala esto automáticamente.
# Localmente: uv pip install -r requirements.txt

# Pipeline RAG
langchain>=1.0.0
langchain-community>=0.4.0
langchain-chroma>=1.0.0
langchain-text-splitters>=1.0.0
langchain-core>=1.0.0
langchain-huggingface>=0.1.0

# Base vectorial y embeddings
chromadb>=0.6.0
sentence-transformers>=3.0.0

# Carga de PDFs
pypdf>=5.0.0

# Interfaz
gradio>=5.0.0

Overwriting requirements.txt


In [37]:
# Verificamos que se generaron correctamente
import os

for archivo in ["app.py", "requirements.txt"]:
    if os.path.exists(archivo):
        tamanio = os.path.getsize(archivo)
        print(f"  ✓ {archivo} — {tamanio} bytes")
    else:
        print(f"  ✗ {archivo} no encontrado")

print("\nEstos dos archivos son todo lo que necesitás subir a tu Space.")

  ✓ app.py — 6874 bytes
  ✓ requirements.txt — 499 bytes

Estos dos archivos son todo lo que necesitás subir a tu Space.


## ⛰️ Cierre del proyecto integrador

### Lo que construiste

| Pieza | Qué hace | Tecnología |
|---|---|---|
| Cargador de PDFs | Extrae texto página por página | PyPDFLoader |
| Divisor de texto | Fragmenta en chunks con overlap | RecursiveCharacterTextSplitter |
| Embeddings | Vectoriza texto localmente | multilingual-e5-large |
| Base vectorial | Almacena y busca fragmentos por similitud | ChromaDB (persistente) |
| LLM local | Genera respuestas fundamentadas | Ollama |
| Pipeline RAG | Conecta todo con LCEL | LangChain |
| Interfaz | Hace el sistema accesible sin código | Gradio |
| Deploy | Publica en la web | HuggingFace Spaces |

### La diferencia entre local y Spaces

| Aspecto | Local | HuggingFace Spaces |
|---|---|---|
| LLM | Ollama (tu hardware) | HuggingFace Inference API |
| ChromaDB | Persistente en disco | En memoria (efímera) |
| Acceso | Solo tu máquina | URL pública |
| Costo | Sin API, solo electricidad | Gratis con cuenta HF |

### 🧭 Diario de Navegación

Cerramos mirando hacia adentro, no hacia el código. Respondé en una o dos líneas:

1. ¿Qué pieza del pipeline te pareció más difícil de entender? ¿Ya la pudiste encajar?
La parte más difícil fue entender cómo el sistema recupera los fragmentos relevantes de los PDF para responder una pregunta. Al principio pensábamos que el modelo leía todo el documento cada vez que recibía una consulta, pero durante las pruebas comprendimos que primero se fragmenta el texto, luego se indexa en ChromaDB y finalmente se recuperan solo los fragmentos más relacionados con la pregunta. También aprendimos que una respuesta incorrecta no siempre significa que el modelo no conozca la información, sino que el sistema puede no haber recuperado los fragmentos adecuados para responder.

2. Si tuvieras que presentar este sistema ante alguien de RRHH de tu empresa, ¿cómo lo explicarías en 30 segundos sin mencionar "embeddings" ni "vectores"?
Lo presentaríamos como un asistente que permite consultar documentos PDF mediante preguntas en lenguaje natural. En lugar de leer manuales completos o buscar información página por página, el usuario puede realizar una consulta específica y recibir una respuesta rápida basada en los documentos cargados, junto con las fuentes utilizadas para generarla.

3. ¿Qué problema real de tu contexto laboral o de estudio resolverías con este sistema?
Este sistema está pensado para ayudar a personas que quieren iniciarse en el mercado de capitales y no conocen los conceptos básicos del tema. En lugar de leer manuales completos o buscar información en distintos documentos, el usuario puede realizar preguntas como "¿Qué es una acción?", "¿Qué es un CEDEAR?" o "¿Qué es una caución?" y obtener una explicación rápida basada en la documentación cargada.
De esta manera, el sistema facilita el aprendizaje de conceptos financieros y permite acceder a la información de forma más simple y directa para quienes están dando sus primeros pasos en el mercado de capitales.

## Conclusión

Se desarrolló un asistente educativo basado en la técnica RAG orientado a la consulta de conceptos del mercado de capitales. El sistema permitió recuperar información desde documentos especializados y generar respuestas comprensibles para usuarios principiantes.

Durante las pruebas se observó que el rendimiento es especialmente bueno cuando las consultas son específicas y están relacionadas con los conceptos presentes en la documentación. Asimismo, se comprobó que el sistema puede realizar búsquedas semánticas incluso cuando las consultas se realizan en inglés y los documentos se encuentran en español.

Como trabajo futuro, el sistema podría ampliarse incorporando nuevos documentos y conceptos para construir un glosario inteligente cada vez más completo sobre mercado de capitales.


## Anexo  - Preguntas guía utilizadas para las pruebas

Durante la validación del sistema se realizaron consultas sobre conceptos básicos del mercado de capitales para evaluar la capacidad de recuperación de información y generación de respuestas.

### Conceptos básicos

* ¿Qué es una acción?
* ¿Qué es un bono?
* ¿Qué es un CEDEAR?
* ¿Qué es una caución?
* ¿Qué es una obligación negociable?
* ¿Qué es una letra?
* ¿Qué es el mercado de capitales?
* ¿Qué es el riesgo financiero?
* ¿Qué es la diversificación?
* ¿Qué es la tasa de interés?

### Instrumentos y perfiles

* ¿Qué perfiles de inversor existen?
* ¿Qué es un portafolio de inversión?
* ¿Qué tipos de inversiones existen en el mercado de capitales?
* ¿Qué es la renta fija?
* ¿Qué es la renta variable?
* ¿Cuál es la diferencia entre una acción y un bono?

### Organismos y operadores

* ¿Qué es la CNV?
* ¿Qué es una sociedad de bolsa?

### Consultas en inglés

* What is financial risk?
* What is a CEDEAR?
* What is a bond?

### Consultas fuera del dominio

* ¿Qué es un perro?
* ¿Qué es un gato?
* ¿Qué es un semáforo?
